In [1]:
import pandas as pd
import ast  # 用来安全地把字符串转成 Python 对象 (比如列表)

# --- 配置参数 ---
INPUT_FILE_PREDICTIONS = 'comment_predictions_pipeline_progress.csv' # 你刚生成的带有预测结果的文件
OUTPUT_FILE_SCORES = 'comment_sentiment_scores.csv' # 保存最终得分的文件

# --- 1. 定义打分函数 ---

def calculate_sentiment_score(predictions_str):
    """
    计算单条评论的情感总分。
    Args:
        predictions_str (str): 包含预测结果列表的字符串，
                               例如 "[{'aspect': '靠近银行', 'sentiment': 'POS'}, ...]"
    Returns:
        int: 该评论的情感总分。
    """
    score = 0
    try:
        # 安全地将字符串转换回列表套字典的格式
        # ast.literal_eval 比 eval() 安全得多
        predictions_list = ast.literal_eval(predictions_str)
        
        if not isinstance(predictions_list, list):
             # 如果转换出来的不是列表，说明格式有问题，返回0分
             print(f"警告：无法解析的预测格式: {predictions_str}")
             return 0

        for aspect_info in predictions_list:
             # 确保 aspect_info 是字典并且包含 'sentiment' 键
            if isinstance(aspect_info, dict) and 'sentiment' in aspect_info:
                sentiment = aspect_info['sentiment']
                if sentiment == 'POS':
                    score += 1
                elif sentiment == 'NEG':
                    score -= 1
                # NEU 的分数为 0，不用加也不用减
            else:
                 print(f"警告：预测列表中的项格式不正确: {aspect_info} in {predictions_str}")

    except (ValueError, SyntaxError) as e:
        # 如果字符串无法被 ast.literal_eval 解析 (比如格式严重错误或为空)
        print(f"错误：解析预测字符串时出错 '{predictions_str}': {e}")
        return 0 # 出错的评论给0分

    return score

# --- 2. 加载数据并应用打分函数 ---

print(f"正在从 {INPUT_FILE_PREDICTIONS} 加载预测结果...")
try:
    df_scores = pd.read_csv(INPUT_FILE_PREDICTIONS)
    
    # 确保 'predictions' 列存在
    if 'predictions' not in df_scores.columns:
        raise ValueError("文件中未找到 'predictions' 列。")
        
    # 处理可能的空值 (NaN)，填充为空列表字符串 '[]'
    df_scores['predictions'] = df_scores['predictions'].fillna('[]')

    print("正在计算每条评论的情感分数...")
    # 应用打分函数，创建新列 'sentiment_score'
    df_scores['sentiment_score'] = df_scores['predictions'].apply(calculate_sentiment_score)
    
    # (可选) 查看一下得分分布
    print("\n情感分数分布概览:")
    print(df_scores['sentiment_score'].value_counts().sort_index())

    # --- 3. 保存带分数的结果 ---
    print(f"\n正在将结果保存到 {OUTPUT_FILE_SCORES} ...")
    df_scores.to_csv(OUTPUT_FILE_SCORES, index=False, encoding='utf-8-sig')
    
    print("\n--- 操作完成 ---")
    print(f"已成功计算情感分数并保存至: {OUTPUT_FILE_SCORES}")

except FileNotFoundError:
    print(f"错误：找不到输入文件 {INPUT_FILE_PREDICTIONS}")
except Exception as e:
    print(f"处理文件时发生错误: {e}")

正在从 comment_predictions_pipeline_progress.csv 加载预测结果...
正在计算每条评论的情感分数...

情感分数分布概览:
sentiment_score
-5        2
-4      397
-3    14384
-2    27120
-1    29475
 0    52405
 1    47658
 2    50217
 3    24536
 4      366
Name: count, dtype: int64

正在将结果保存到 comment_sentiment_scores.csv ...

--- 操作完成 ---
已成功计算情感分数并保存至: comment_sentiment_scores.csv
